In [20]:
import pandas as pd
import numpy as np

orig_cols = [
    'credit_score','first_payment_date','first_time_homebuyer_flag','maturity_date',
    'msa','mi_pct','number_of_units','occupancy_status','orig_cltv','orig_dti',
    'orig_upb','orig_ltv','orig_interest_rate','channel','ppm_flag','amortization_type',
    'property_state','property_type','postal_code','loan_seq_no','loan_purpose',
    'orig_loan_term','number_of_borrowers','seller_name','servicer_name',
    'super_conforming_flag','pre_harp_loan_seq','program_indicator','harp_indicator',
    'property_valuation_method','interest_only_indicator','mi_cancellation_indicator'
]

orig = pd.read_csv('../data/raw/sample_orig_2007.txt', sep='|',
                   names=orig_cols, header=None, dtype=str, keep_default_na=False)
orig.head()

,credit_score,first_payment_date,first_time_homebuyer_flag,maturity_date,msa,mi_pct,number_of_units,occupancy_status,orig_cltv,orig_dti,...,number_of_borrowers,seller_name,servicer_name,super_conforming_flag,pre_harp_loan_seq,program_indicator,harp_indicator,property_valuation_method,interest_only_indicator,mi_cancellation_indicator
0,769,200702,N,203701,24220,000,1,P,90,38,...,02,Other sellers,Other servicers,,,9,,7,N,9
1,598,200703,N,203702,,000,1,P,90,45,...,02,Other sellers,Other servicers,,,9,,7,N,9
2,777,200703,N,203702,20260,000,1,P,75,23,...,02,Other sellers,U.S. BANK N.A.,,,9,,7,N,9
3,628,200703,N,203702,35644,000,1,P,65,38,...,02,Other sellers,Other servicers,,,9,,7,N,9
4,696,200703,N,203702,,000,1,P,80,23,...,02,Other sellers,Other servicers,,,9,,7,N,9


In [11]:
numeric_cols = ['credit_score','orig_cltv','orig_dti','orig_upb','orig_ltv',
                'orig_interest_rate','orig_loan_term','number_of_units','mi_pct']
for c in numeric_cols:
    orig[c] = pd.to_numeric(orig[c], errors='coerce')

In [12]:
orig.info()                    # column types and non-null counts
orig.describe()                # ranges — spot impossible values
orig.isna().sum()              # missing counts per column
orig['credit_score'].value_counts().sort_index()   # look for sentinel codes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 32 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   credit_score               50000 non-null  int64  
 1   first_payment_date         50000 non-null  object 
 2   first_time_homebuyer_flag  50000 non-null  object 
 3   maturity_date              50000 non-null  object 
 4   msa                        50000 non-null  object 
 5   mi_pct                     50000 non-null  int64  
 6   number_of_units            50000 non-null  int64  
 7   occupancy_status           50000 non-null  object 
 8   orig_cltv                  50000 non-null  int64  
 9   orig_dti                   50000 non-null  int64  
 10  orig_upb                   50000 non-null  int64  
 11  orig_ltv                   50000 non-null  int64  
 12  orig_interest_rate         50000 non-null  float64
 13  channel                    50000 non-null  obj

credit_score
300      1
379      1
448      1
474      1
480      1
        ..
832      2
834      1
837      4
844      1
9999    38
Name: count, Length: 328, dtype: int64

In [13]:
orig.loc[orig['credit_score'] == 9999, 'credit_score'] = np.nan   # 38 rows
orig.loc[orig['orig_dti']    == 999,  'orig_dti']      = np.nan   # 1,124 rows
orig.loc[orig['orig_cltv']   == 999,  'orig_cltv']     = np.nan   # 2 rows
orig.loc[orig['orig_ltv']    == 999,  'orig_ltv']      = np.nan   # 1 row

In [14]:
orig['first_payment_date'] = pd.to_datetime(orig['first_payment_date'], format='%Y%m')
orig['maturity_date']      = pd.to_datetime(orig['maturity_date'],      format='%Y%m')

In [21]:
orig = orig.drop(columns=['super_conforming_flag','pre_harp_loan_seq','harp_indicator'])

In [23]:
perf_cols = [
    'loan_seq_no','reporting_period','current_upb','delinquency_status','loan_age',
    'months_to_maturity','defect_settlement_date','modification_flag','zero_balance_code',
    'zero_balance_date','current_interest_rate','current_deferred_upb','ddlpi','mi_recoveries',
    'net_sales_proceeds','non_mi_recoveries','expenses','legal_costs','maint_costs',
    'taxes_insurance','misc_expenses','actual_loss','modification_cost','step_mod_flag',
    'deferred_payment_plan','eltv','zb_removal_upb','delinquent_accrued_interest',
    'delinquency_disaster','borrower_assistance','month_mod_cost','interest_bearing_upb'
]

# Only read the 3 columns we need — the full file is 250+ MB
perf = pd.read_csv('../data/raw/sample_svcg_2007.txt', sep='|', names=perf_cols,
                   header=None, dtype=str, keep_default_na=False,
                   usecols=['loan_seq_no','delinquency_status','zero_balance_code'])

# delinquency as a number; 'RA','XX', blanks become NaN
perf['dq'] = pd.to_numeric(perf['delinquency_status'], errors='coerce')

# three default signals, collapsed to ONE row per loan
ever_90plus   = perf.groupby('loan_seq_no')['dq'].max() >= 3
ever_reo      = perf['delinquency_status'].eq('RA').groupby(perf['loan_seq_no']).any()
credit_event  = perf['zero_balance_code'].isin(['03','09']).groupby(perf['loan_seq_no']).any()

defaulted = (ever_90plus | ever_reo | credit_event).astype(int).rename('defaulted')
print(f"Default rate: {defaulted.mean()*100:.2f}%  ({defaulted.sum():,} of {len(defaulted):,})")

Default rate: 16.52%  (8,258 of 50,000)


In [25]:
final = orig.merge(defaulted, on='loan_seq_no', how='left')
final['defaulted'] = final['defaulted'].fillna(0).astype(int)  # safety: any loan with no perf rows = no default observed

final.to_parquet('../data/processed/loans_2007_clean.parquet')
print(final.shape)   # (50000, 30)

(50000, 30)


In [28]:
import pandas as pd

perf_cols = ['loan_seq_no','reporting_period','current_upb','delinquency_status','loan_age',
    'months_to_maturity','defect_settlement_date','modification_flag','zero_balance_code',
    'zero_balance_date','current_interest_rate','current_deferred_upb','ddlpi','mi_recoveries',
    'net_sales_proceeds','non_mi_recoveries','expenses','legal_costs','maint_costs','taxes_insurance',
    'misc_expenses','actual_loss','modification_cost','step_mod_flag','deferred_payment_plan','eltv',
    'zb_removal_upb','delinquent_accrued_interest','delinquency_disaster','borrower_assistance',
    'month_mod_cost','interest_bearing_upb']

perf = pd.read_csv('../data/raw/sample_svcg_2007.txt', sep='|', names=perf_cols, header=None,
                   dtype=str, keep_default_na=False,
                   usecols=['loan_seq_no','reporting_period','current_upb','delinquency_status',
                            'loan_age','zero_balance_code','current_interest_rate'])

perf['reporting_period'] = pd.to_datetime(perf['reporting_period'], format='%Y%m')
perf['current_upb']           = pd.to_numeric(perf['current_upb'], errors='coerce')
perf['current_interest_rate'] = pd.to_numeric(perf['current_interest_rate'], errors='coerce')
perf['loan_age']              = pd.to_numeric(perf['loan_age'], errors='coerce').astype('Int64')

perf.to_parquet('../data/processed/performance_2007.parquet')
print(perf.shape)   # expect (3003932, 7)

(3003932, 7)


In [31]:
%pip install python-dotenv sqlalchemy psycopg2-binary

  Using cached psycopg2_binary-2.9.12-cp313-cp313-win_amd64.whl.metadata (5.1 kB)
Using cached psycopg2_binary-2.9.12-cp313-cp313-win_amd64.whl (2.8 MB)
Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
from pathlib import Path
from dotenv import load_dotenv

env_path = Path("../.env").resolve()

print("ENV path:", env_path)
print("ENV exists:", env_path.exists())

load_dotenv(env_path, override=True)

print("DB_HOST:", os.getenv("DB_HOST"))
print("DB_PORT:", os.getenv("DB_PORT"))
print("DB_NAME:", os.getenv("DB_NAME"))
print("DB_USER:", os.getenv("DB_USER"))
print("DB_PASSWORD exists:", os.getenv("DB_PASSWORD") is not None)

ENV path: C:\Users\User\OneDrive\Desktop\Portfolio\Credit risk\credit-risk-insight-engine\.env
ENV exists: True
DB_HOST: localhost
DB_PORT: 5432
DB_NAME: credit_risk
DB_USER: postgres
DB_PASSWORD exists: True


In [3]:
import os
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

# Load .env from the main project folder
env_path = Path("../.env")
load_dotenv(env_path)

url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    database=os.getenv("DB_NAME")
)

engine = create_engine(url)

with engine.connect() as c:
    result = c.execute(text("SELECT 1")).scalar()
    print(result)

1


In [13]:
import pandas as pd
from sqlalchemy import text

loans = pd.read_parquet("../data/processed/loans_2007_clean.parquet")

date_cols = ["first_payment_date", "maturity_date"]

for col in date_cols:
    loans[col] = pd.to_datetime(
        loans[col].astype(str).str.extract(r"(\d{6})", expand=False) + "01",
        format="%Y%m%d",
        errors="coerce"
    ).dt.date

# Check that dates are fixed before loading
print(loans[date_cols].head())
print(loans[date_cols].dtypes)

# Optional: clear table before loading again
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE loans CASCADE;"))

loans.to_sql("loans", engine, if_exists="append", index=False)

print("loaded", len(loans), "loans")

  first_payment_date maturity_date
0         2007-02-01    2037-01-01
1         2007-03-01    2037-02-01
2         2007-03-01    2037-02-01
3         2007-03-01    2037-02-01
4         2007-03-01    2037-02-01
first_payment_date    object
maturity_date         object
dtype: object
loaded 50000 loans


In [16]:
perf = pd.read_parquet("../data/processed/performance_2007.parquet")

print(perf.columns.tolist())
print(perf["reporting_period"].head(10))
print(perf["reporting_period"].dtype)
print(perf["reporting_period"].isna().sum())

['loan_seq_no', 'reporting_period', 'current_upb', 'delinquency_status', 'loan_age', 'zero_balance_code', 'current_interest_rate']
0   2007-01-01
1   2007-02-01
2   2007-03-01
3   2007-04-01
4   2007-05-01
5   2007-06-01
6   2007-07-01
7   2007-08-01
8   2007-09-01
9   2007-10-01
Name: reporting_period, dtype: datetime64[ns]
datetime64[ns]
0


In [17]:
import os
import io
import time
import pandas as pd
import psycopg2
from pathlib import Path
from dotenv import load_dotenv
from sqlalchemy import text

load_dotenv(Path("../.env"), override=True)

perf = pd.read_parquet("../data/processed/performance_2007.parquet")

perf_cols = [
    "loan_seq_no",
    "reporting_period",
    "current_upb",
    "delinquency_status",
    "loan_age",
    "zero_balance_code",
    "current_interest_rate"
]

perf = perf[perf_cols].copy()

# Robust reporting_period conversion
raw_period = perf["reporting_period"]

if pd.api.types.is_datetime64_any_dtype(raw_period):
    perf["reporting_period"] = raw_period.dt.date
else:
    period_str = raw_period.astype(str).str.strip()

    # Handles values like 200702.0
    period_str = period_str.str.replace(r"\.0$", "", regex=True)

    # If value is YYYYMM, convert to YYYYMM01
    yyyymm = period_str.str.extract(r"^(\d{6})$", expand=False)
    perf["reporting_period"] = pd.to_datetime(
        yyyymm + "01",
        format="%Y%m%d",
        errors="coerce"
    ).dt.date

    # If some values were already normal dates like 2007-02-01, try parsing those too
    missing_mask = perf["reporting_period"].isna()
    perf.loc[missing_mask, "reporting_period"] = pd.to_datetime(
        period_str[missing_mask],
        errors="coerce"
    ).dt.date

# Check before loading
print(perf[["loan_seq_no", "reporting_period"]].head())
print("Null reporting_period count:", perf["reporting_period"].isna().sum())

# Stop if reporting_period still has nulls
if perf["reporting_period"].isna().sum() > 0:
    bad_rows = perf[perf["reporting_period"].isna()].head(10)
    display(bad_rows)
    raise ValueError("reporting_period still has nulls. Fix these before loading.")

# Clear old data before retrying
with engine.begin() as conn:
    conn.execute(text("TRUNCATE TABLE loan_performance;"))

t = time.time()

buf = io.StringIO()
perf.to_csv(
    buf,
    index=False,
    header=False,
    na_rep=r"\N"
)
buf.seek(0)

raw = psycopg2.connect(
    host=os.getenv("DB_HOST"),
    port=os.getenv("DB_PORT"),
    dbname=os.getenv("DB_NAME"),
    user=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD")
)

cur = raw.cursor()

cur.copy_expert(
    """
    COPY loan_performance
    (
        loan_seq_no,
        reporting_period,
        current_upb,
        delinquency_status,
        loan_age,
        zero_balance_code,
        current_interest_rate
    )
    FROM STDIN WITH (FORMAT csv, NULL '\\N')
    """,
    buf
)

raw.commit()
cur.close()
raw.close()

print(f"loaded performance in {time.time() - t:.0f}s")
print("rows loaded:", len(perf))

    loan_seq_no reporting_period
0  F07Q10000023       2007-01-01
1  F07Q10000023       2007-02-01
2  F07Q10000023       2007-03-01
3  F07Q10000023       2007-04-01
4  F07Q10000023       2007-05-01
Null reporting_period count: 0
loaded performance in 123s
rows loaded: 3003932


In [18]:
with engine.begin() as c:
    c.execute(text("ALTER TABLE loan_performance ADD PRIMARY KEY (loan_seq_no, reporting_period)"))
    c.execute(text("CREATE INDEX idx_perf_period  ON loan_performance(reporting_period)"))
    c.execute(text("CREATE INDEX idx_loans_credit ON loans(credit_score)"))
    c.execute(text("CREATE INDEX idx_loans_state  ON loans(property_state)"))

In [19]:
with engine.connect() as c:
    print("loans:", c.execute(text("SELECT COUNT(*) FROM loans")).scalar())
    print("perf :", c.execute(text("SELECT COUNT(*) FROM loan_performance")).scalar())
    print("default rate:", c.execute(text("SELECT ROUND(100.0*AVG(defaulted),2) FROM loans")).scalar())
    orphans = c.execute(text("""SELECT COUNT(*) FROM loan_performance p
                                LEFT JOIN loans l ON p.loan_seq_no=l.loan_seq_no
                                WHERE l.loan_seq_no IS NULL""")).scalar()
    print("orphans:", orphans)

loans: 50000
perf : 3003932
default rate: 16.52
orphans: 0
